# CampusAssist: RAG-Based University Helpdesk Support Chatbot
## CSE 476 Project: Conversational RAG with Grounded Responses & Memory

This notebook demonstrates the complete implementation and evaluation of **CampusAssist**, a Retrieval-Augmented Generation (RAG) conversational chatbot specifically designed for a **University Helpdesk** support role.

### Core Architectural Pillars:
1. **Curated Knowledge Base**: 16 structured university FAQ pairs covering IT, library, registrar, billing, dorms, advising, and health.
2. **Hybrid RAG Retriever**: Combines dense semantic vector embeddings (`all-MiniLM-L6-v2`) with BM25 keyword matching and cosine similarity scoring.
3. **Conversational Memory**: Maintains multi-turn context and performs **Contextual Query Reformulation** to resolve pronouns and ellipsis in follow-up queries.
4. **Strict Grounding Guardrails**: Evaluates retrieval confidence against a strict threshold to eliminate hallucinations and explicitly disclaim unavailable information.
5. **Multi-LLM & Local Engine**: Supports Groq (Llama / Qwen), OpenAI, Gemini, and a built-in Local Grounded Engine.

### 1. Environment Setup and Imports

In [ ]:
import os
import json
from dotenv import load_dotenv
load_dotenv()

from src.knowledge_base import KnowledgeBase
from src.retriever import HybridRetriever
from src.memory import ConversationMemory
from src.generator import GroundedGenerator, UNAVAILABLE_DISCLAIMER
from src.rag_pipeline import RAGPipeline

print("All modules imported successfully!")

### 2. Knowledge Base Inspection
Load and view the university helpdesk FAQ dataset (16 Q&A items across campus services).

In [ ]:
kb = KnowledgeBase()
print(f"Total FAQ Items in Knowledge Base: {len(kb)}")
print(f"Categories: {kb.get_categories()}\n")

# Display first 3 sample items
for item in kb.get_all()[:3]:
    print(f"[{item.id}] ({item.category})")
    print(f"Q: {item.question}")
    print(f"A: {item.answer[:120]}...")
    print("-" * 60)

### 3. Hybrid Vector Retrieval with Confidence Thresholding
Test dense vector similarity + keyword BM25 retrieval, and verify score separation between valid support questions and unrelated out-of-domain queries.

In [ ]:
retriever = HybridRetriever(
    chunks=kb.get_chunks_for_indexing(),
    relevance_threshold=0.45
)

# Test 1: In-domain question
query_in = "How do I connect to campus Wi-Fi?"
results_in, is_rel_in = retriever.retrieve(query_in, top_k=2)
print(f"Query: '{query_in}'")
print(f"Is Relevant: {is_rel_in} (Top Score: {results_in[0].score:.4f})")
print(f"Top Chunk Matched: [{results_in[0].id}] {results_in[0].question}\n")

# Test 2: Out-of-domain question
query_out = "How do I bake chocolate chip cookies?"
results_out, is_rel_out = retriever.retrieve(query_out, top_k=2)
print(f"Query: '{query_out}'")
print(f"Is Relevant: {is_rel_out} (Top Score: {results_out[0].score:.4f})")
print("Notice: The score is well below the 0.45 threshold, correctly flagging it as irrelevant!")

### 4. Conversational Memory & Contextual Query Reformulation
When a user asks a follow-up question containing pronouns (e.g., "What about guests?", "How much does that cost?"), the memory module detects the dependency and reformulates it into a standalone query containing the subject topic.

In [ ]:
memory = ConversationMemory()

# Simulate Turn 1
memory.add_user_message("How do I get a replacement student ID card?")
memory.add_assistant_message("Visit the Campus Card Office in Student Center Room 105. A $20 fee applies.")

# Turn 2: Follow-up query
followup_q = "How much does that cost and where do I pick it up?"
print(f"Follow-up check: {memory.is_followup_query(followup_q)}")

standalone_q = memory.reformulate_query(followup_q)
print(f"Original Query:     '{followup_q}'")
print(f"Reformulated Query: '{standalone_q}'")

### 5. End-to-End Conversational RAG Demonstration
Run a 5-turn conversation through `RAGPipeline` demonstrating:
1. Direct in-domain question
2. Follow-up query with pronoun resolution
3. New topic query
4. Follow-up query with ellipsis
5. Out-of-domain query handled with grounded fallback disclaimer

In [ ]:
pipeline = RAGPipeline(
    generator_provider="auto", # Auto uses Groq if GROQ_API_KEY is available, or Local Engine
    relevance_threshold=0.45,
    top_k=3
)

conversation = [
    "How do I connect to the university Wi-Fi network?",
    "What about guests who do not have student credentials?",
    "How do I get a replacement student ID card?",
    "How much does that cost and where do I pick it up?",
    "What is the recipe for baking chocolate chip cookies?"
]

for i, user_q in enumerate(conversation, 1):
    print(f"\n{'='*70}")
    print(f"TURN {i}: USER: \"{user_q}\"")
    print('='*70)
    
    res = pipeline.process_query(user_q)
    
    if res["is_followup"]:
        print(f"[Contextual Search Query: '{res['search_query']}']")
    print(f"[Relevance: {res['is_relevant']} | Confidence Score: {res['confidence_score']:.4f}]")
    print(f"\nASSISTANT:\n{res['response']}")

### 6. Summary of Key Achievements
- **Domain Alignment**: High-fidelity University Helpdesk support system addressing real campus operational workflows.
- **Precision Retrieval**: Hybrid dense-semantic vector retrieval paired with BM25 keyword matching for exact acronyms (`eduroam`, `MFA`, `ID`, `Wi-Fi`).
- **Multi-Turn Context Tracking**: Automatic coreference resolution preserving conversational continuity across turns.
- **Grounded Output Guarantee**: Strict relevance gating prevents hallucinations and clearly informs users when information is unavailable.